# DocuDevs: Pipeline Extraction

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/docudevs/python-examples/blob/main/05-pipeline-extraction.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-View_Source-blue?logo=github)](https://github.com/docudevs/python-examples)

Build a document-processing pipeline that first classifies a document, then routes to the right extraction branch.

**What you'll learn:**
- How to define a pipeline with the SDK builder
- How to classify once and branch with `when` conditions
- How to inspect node status and skipped branches

In [ ]:
# Install dependencies (required in Colab)
# Colab includes packages such as gradio and google-genai with their own
# pydantic/httpx constraints. Install compatible shared deps first, then
# install the SDK without letting older SDK metadata downgrade them.
%pip install -q --upgrade "pydantic>=2.0,<=2.12.3" "httpx>=0.28.1,<1.0" "attrs>=21.3.0" "python-dateutil>=2.8.0" "click>=8.0.0"
%pip install -q --upgrade --no-deps docu-devs-api-client

In [ ]:
import json
import os
from pathlib import Path
from typing import Literal, Optional
import urllib.request

from docudevs import DocuDevsClient, P, Pipeline, eq
from pydantic import BaseModel, Field

try:
    from google.colab import userdata
    API_KEY = userdata.get("DOCUDEVS_API_KEY")
except Exception:
    API_KEY = os.getenv("DOCUDEVS_API_KEY", "your-api-key-here")

if not API_KEY or API_KEY == "your-api-key-here":
    raise ValueError("Set your DOCUDEVS_API_KEY - get one at https://docudevs.ai")

client = DocuDevsClient(token=API_KEY)
print("Connected to DocuDevs API")

Connected to DocuDevs API


## Pipeline Shape

This pipeline has one shared OCR pass and three logical phases:

1. `classify_document` decides whether the document is an `invoice`, `safety_data_sheet`, or `other`.
2. `extract_invoice` and `extract_sds` are conditional branches. Only the matching branch runs.
3. Final candidates return a normalized result with the classification and branch extraction.

The SDK builder gives you typed node references (`NodeRef`) and path helpers (`P.ocr.content`, `node.result`) so you do not have to hand-write strings such as `$nodes.classify_document.result`.

In [ ]:
class DocumentClassification(BaseModel):
    document_type: Literal["invoice", "safety_data_sheet", "other"] = Field(
        description="The best matching document type. Use exactly one allowed value."
    )
    confidence: float = Field(description="Classifier confidence from 0 to 1", ge=0, le=1)
    reason: str = Field(description="Short explanation of the classification decision")


class InvoiceExtraction(BaseModel):
    invoice_number: Optional[str] = None
    vendor_name: Optional[str] = None
    invoice_date: Optional[str] = None
    due_date: Optional[str] = None
    total_amount: Optional[float] = None
    currency: Optional[str] = None
    payment_terms: Optional[str] = None


class SafetyDataSheetExtraction(BaseModel):
    product_name: Optional[str] = None
    supplier: Optional[str] = None
    signal_word: Optional[str] = None
    hazard_classification: Optional[list[str]] = None
    first_aid_summary: Optional[str] = None
    handling_and_storage: Optional[str] = None


classification_prompt = """
Classify this document. Return document_type exactly as one of:
- invoice: an invoice, bill, receipt, or payment request with totals or payment terms
- safety_data_sheet: an SDS/MSDS or chemical product safety document
- other: anything else

Do not extract detailed business fields in this step. Only classify the document.
""".strip()

invoice_prompt = """
Extract invoice fields from the document using the schema.
Return null for fields that are genuinely missing. Do not infer or invent values.
""".strip()

sds_prompt = """
Extract product and safety information from the document using the schema.
Summaries should be concise and based only on the document text.
Return null for fields that are genuinely missing.
""".strip()


document_router_pipeline = Pipeline().ocr(mode="AUTO", quality_artifact=True).max_nodes(2)

classify_document = document_router_pipeline.extract(
    "classify_document",
    source=P.ocr.content,
    prompt=classification_prompt,
    schema=DocumentClassification,
    llm_tier="nano",
)
extract_invoice = document_router_pipeline.extract(
    "extract_invoice",
    depends_on=[classify_document],
    when=eq(classify_document.result.document_type, "invoice"),
    source=P.ocr.content,
    prompt=invoice_prompt,
    schema=InvoiceExtraction,
    llm_tier="mini",
)
extract_sds = document_router_pipeline.extract(
    "extract_sds",
    depends_on=[classify_document],
    when=eq(classify_document.result.document_type, "safety_data_sheet"),
    source=P.ocr.content,
    prompt=sds_prompt,
    schema=SafetyDataSheetExtraction,
    llm_tier="mini",
)
invoice_final = document_router_pipeline.final_candidate(
    "invoice_final",
    depends_on=[classify_document, extract_invoice],
    when=eq(classify_document.result.document_type, "invoice"),
    output={
        "branch": "invoice",
        "classification": classify_document.result,
        "extracted": extract_invoice.result,
    },
)
sds_final = document_router_pipeline.final_candidate(
    "sds_final",
    depends_on=[classify_document, extract_sds],
    when=eq(classify_document.result.document_type, "safety_data_sheet"),
    output={
        "branch": "safety_data_sheet",
        "classification": classify_document.result,
        "extracted": extract_sds.result,
    },
)
other_final = document_router_pipeline.final_candidate(
    "other_final",
    depends_on=[classify_document],
    when=eq(classify_document.result.document_type, "other"),
    output={
        "branch": "other",
        "classification": classify_document.result,
        "message": "No specialized extraction branch was selected.",
    },
)
document_router_pipeline.final_order(invoice_final, sds_final, other_final)

print(json.dumps(document_router_pipeline.to_dict(), indent=2)[:2000] + "\n...")

## Load Sample Documents

The invoice run is the default example. The safety data sheet run later is optional because it starts a second pipeline job.

In [3]:
Path("docs").mkdir(exist_ok=True)

sample_files = ["invoice.pdf", "samplepreppro-msds.pdf"]
for name in sample_files:
    path = Path("docs") / name
    if not path.exists():
        urllib.request.urlretrieve(
            f"https://raw.githubusercontent.com/docudevs/python-examples/main/docs/{name}",
            path,
        )
        print(f"Downloaded {name}")

with open("docs/invoice.pdf", "rb") as f:
    invoice_bytes = f.read()

with open("docs/samplepreppro-msds.pdf", "rb") as f:
    sds_bytes = f.read()

print(f"Loaded invoice.pdf: {len(invoice_bytes):,} bytes")
print(f"Loaded samplepreppro-msds.pdf: {len(sds_bytes):,} bytes")

Loaded invoice.pdf: 2,691,626 bytes
Loaded samplepreppro-msds.pdf: 2,080,685 bytes


## Run the Pipeline on an Invoice

In [4]:
async def run_document_router_pipeline(document_bytes, mime_type="application/pdf"):
    job_id = await client.process_pipeline_document(
        document=document_bytes,
        document_mime_type=mime_type,
        pipeline=document_router_pipeline,
        ocr="AUTO",
        trace=True,
    )
    result = await client.wait_until_ready(
        job_id,
        timeout=900,
        poll_interval=5,
        result_format="json",
    )
    return job_id, result


invoice_job_id, invoice_result = await run_document_router_pipeline(invoice_bytes)

print(f"Pipeline job completed: {invoice_job_id}")
print(json.dumps(invoice_result, indent=2))

Pipeline job completed: d9cbe0a8-9143-4568-bc8e-e08ca2793bab
{
  "branch": "invoice",
  "classification": {
    "document_type": "invoice",
    "confidence": 0.93,
    "reason": "The document contains an invoice with line items, subtotal, VAT, an invoice total (CHF 50.00), an invoice number, and payment terms (transfer within 30 days), along with a QR-bill payment section."
  },
  "extracted": {
    "invoice_number": "999",
    "vendor_name": "Max Muster & S\u00f6hne",
    "invoice_date": "2021-01-15",
    "due_date": null,
    "total_amount": 50.0,
    "currency": "CHF",
    "payment_terms": "within 30 days"
  }
}


## Inspect Node Status

Node status is the easiest way to see the routing decision. For an invoice, the invoice branch should complete while the SDS branch is skipped.

In [5]:
def print_pipeline_nodes(nodes):
    for node in nodes:
        node_id = node.get("id", "unknown")
        status = node.get("status", "unknown")
        child_guid = node.get("childOperationGuid")
        suffix = f" child={child_guid}" if child_guid else ""
        print(f"{node_id:20} {status}{suffix}")


invoice_nodes = await client.get_pipeline_nodes(invoice_job_id)
print_pipeline_nodes(invoice_nodes)

classify_document    COMPLETED
extract_invoice      COMPLETED
extract_sds          SKIPPED
invoice_final        COMPLETED
sds_final            SKIPPED
other_final          SKIPPED


## Optional: Run the Same Pipeline on a Safety Data Sheet

The exact same pipeline can route a different document to a different extraction branch. Set `RUN_SDS_EXAMPLE = True` if you want to submit a second job.

In [7]:
RUN_SDS_EXAMPLE = False

if RUN_SDS_EXAMPLE:
    sds_job_id, sds_result = await run_document_router_pipeline(sds_bytes)
    print(f"Pipeline job completed: {sds_job_id}")
    print(json.dumps(sds_result, indent=2))

    print("\nNode status:")
    sds_nodes = await client.get_pipeline_nodes(sds_job_id)
    print_pipeline_nodes(sds_nodes)
else:
    print("Set RUN_SDS_EXAMPLE = True to run the safety data sheet branch.")

Pipeline job completed: f3de7bdd-cea7-4b68-bd08-6515f0aa8513
{
  "branch": "safety_data_sheet",
  "classification": {
    "document_type": "safety_data_sheet",
    "confidence": 0.99,
    "reason": "The document is a Generic Safety Data Sheet (SDS) cover sheet and includes detailed SDS sections (e.g., hazards identification, first aid, accidental release, handling, storage, transport, regulatory information) for Alkylation Solution/2-Iodoacetamide."
  },
  "extracted": {
    "product_name": "Alkylation Solution",
    "supplier": "Biognosys AG",
    "signal_word": "Danger",
    "hazard_classification": [
      "Acute toxicity (oral) - Category 3",
      "Respiratory sensitization - Category 1",
      "Skin sensitization - Category 1",
      "Aquatic hazard (long-term) - Category 4"
    ],
    "first_aid_summary": "Eye: flush with water for at least 20 minutes and get medical attention if irritation occurs. Inhalation: move to fresh air; get medical attention if symptoms occur or breathi

## When to Use Pipeline Mode

Use pipeline mode when the later steps depend on earlier decisions:

- Classify first, then use the right schema and prompt.
- Run quality checks before expensive extraction branches.
- Keep one workflow auditable by inspecting node status and skipped branches.

If every extraction can run independently on the same OCR text, `STEPS` mode may be simpler. If the document is large and needs chunking, map-reduce is usually the better starting point.